In [2]:
import numpy as np
import pandas as pd

In [10]:
df_train = pd.read_csv("../data/train_raw.csv")

In [11]:
print("data size : ",df_train.shape)
df_train.describe().T

data size :  (132937, 24)


,count,mean,std,min,25%,50%,75%,max
timestamp_ns,132937.0,1.536055e+18,2.790122e+15,1.531944e+18,1.532985e+18,1.536693e+18,1.538765e+18,1.539876e+18
airspeed_cmd,132937.0,1.562505e+01,4.834310e-01,1.500000e+01,1.500000e+01,1.600000e+01,1.600000e+01,1.600000e+01
airspeed_meas,132937.0,4.893964e+00,7.405477e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.448553e+01,2.648424e+01
airspeed_error,132937.0,1.073109e+01,7.322463e+00,-1.133438e+01,1.420175e+00,1.500000e+01,1.600000e+01,1.600000e+01
roll_cmd,132937.0,4.774754e+00,2.023759e+01,-4.500000e+01,-2.340000e+00,1.170000e+00,1.375000e+01,4.499000e+01
roll_meas,132937.0,6.370051e+00,1.865174e+01,-6.645238e+01,-1.124623e+00,2.366713e+00,1.384934e+01,7.128139e+01
roll_error,132937.0,-1.595298e+00,1.245943e+01,-1.062053e+02,-5.706806e+00,-8.780489e-01,3.903264e+00,9.828865e+01
pitch_cmd,132914.0,-6.528449e-01,5.751668e+00,-2.499000e+01,-3.690000e+00,-4.200000e-01,1.910000e+00,2.000000e+01
pitch_meas,132914.0,6.437144e-01,6.690156e+00,-5.688078e+01,-1.615566e+00,5.730504e-01,2.629524e+00,7.293653e+01
pitch_error,132914.0,-1.296559e+00,7.204381e+00,-9.304937e+01,-4.185391e+00,-1.114845e+00,1.695808e+00,5.941078e+01


In [8]:
df.isnull().sum()

bag                  0
timestamp_ns         0
airspeed_cmd         0
airspeed_meas        0
airspeed_error       0
roll_cmd             0
roll_meas            0
roll_error           0
pitch_cmd           31
pitch_meas          31
pitch_error         31
yaw_cmd             29
yaw_meas            29
yaw_error           29
vel_x             8332
vel_y             8332
vel_z             8332
imu_x                0
imu_y                0
imu_z                0
engine_fault         0
aileron_fault        0
elevator_fault       0
rudder_fault         0
dtype: int64

In [ ]:
def preprocess_pipeline(df_raw, gap_ns_threshold=1000 * 1e9):
  
    print("Starting preprocessing pipeline...")
    df = df_raw.copy()
    
    # --- 1. Chronological Sorting ---
    df = df.sort_values('timestamp_ns').reset_index(drop=True)
    
    # --- 2. Flight Segmentation (Bag + Time Gap Detection) ---
    df['time_gap_ns'] = df.groupby('bag', sort=False)['timestamp_ns'].diff()
    bag_changed = df['bag'] != df['bag'].shift(1)
    large_gap = df['time_gap_ns'] > gap_ns_threshold
    
    df['flight_id'] = (bag_changed | large_gap).cumsum() - 1
    df.drop(columns=['time_gap_ns'], inplace=True)
    
    # --- 3. Compute Per-Flight Time Metrics ---
    df['time_sec'] = (df['timestamp_ns'] - df['timestamp_ns'].min()) / 1e9  # Global time in sec
    df['flight_time_sec'] = df.groupby('flight_id', sort=False)['timestamp_ns'].transform(
        lambda x: (x - x.min()) / 1e9
    )
    df['dt'] = df.groupby('flight_id', sort=False)['flight_time_sec'].diff()
    
    # --- 4. Yaw Error Angle Wrapping ---
    if 'yaw_error' in df.columns:
        df['yaw_error_clean'] = (df['yaw_error'] + 180) % 360 - 180
        
    # --- 5. Ground Speed Calculation ---
    if {'vel_x', 'vel_y', 'vel_z'}.issubset(df.columns):
        df['ground_speed'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    
    # --- 6. Kinematic Derivatives (Safe per flight_id) ---
    safe_dt = df['dt'].replace(0, np.nan)
    
    # Tracking Error Rates
    if 'airspeed_error' in df.columns:
        df['d_airspeed_err_dt'] = df.groupby('flight_id', sort=False)['airspeed_error'].diff() / safe_dt
    if 'roll_error' in df.columns:
        df['d_roll_err_dt'] = df.groupby('flight_id', sort=False)['roll_error'].diff() / safe_dt
    if 'pitch_error' in df.columns:
        df['d_pitch_err_dt'] = df.groupby('flight_id', sort=False)['pitch_error'].diff() / safe_dt
    if 'yaw_error_clean' in df.columns:
        df['d_yaw_err_dt'] = df.groupby('flight_id', sort=False)['yaw_error_clean'].diff() / safe_dt
        
    # Physical Measurement Rates
    if 'airspeed_meas' in df.columns:
        df['d_airspeed_meas_dt'] = df.groupby('flight_id', sort=False)['airspeed_meas'].diff() / safe_dt
    df['d_roll_meas_dt'] = df.groupby('flight_id', sort=False)['roll_meas'].diff() / safe_dt
    df['d_pitch_meas_dt'] = df.groupby('flight_id', sort=False)['pitch_meas'].diff() / safe_dt
    df['d_yaw_meas_dt'] = df.groupby('flight_id', sort=False)['yaw_meas'].diff() / safe_dt
    
    # Ground Acceleration
    if 'ground_speed' in df.columns:
        df['ground_accel'] = df.groupby('flight_id', sort=False)['ground_speed'].diff() / safe_dt

    # --- 7. Target Labels ---
    fault_cols = ['engine_fault', 'aileron_fault', 'elevator_fault', 'rudder_fault']
    present_faults = [c for c in fault_cols if c in df.columns]
    if present_faults:
        df['binary_fault'] = (df[present_faults] > 0).any(axis=1).astype(int)

    # --- 8. Fill Boundary NaNs Safely ---
    derivative_cols = [col for col in df.columns if col.startswith('d_') or col == 'dt' or col == 'ground_accel']
    df[derivative_cols] = df[derivative_cols].fillna(0.0)

    print(f"Pipeline complete!")
    print(f"Output shape: {df.shape}")
    print(f"Total flight segments detected: {df['flight_id'].nunique()}")
    
    return df